# 🎓 Agente de Reintegros

**Flujo:** el usuario solicita un reintegro → el agente busca en la base de facturación si
corresponde → ejecuta las devoluciones que puede ejecutar solo y **reserva al humano** las que no.

## Dos fuentes de datos

| Dataset | Qué es | Quién lo escribe |
|---|---|---|
| `solicitudes.csv` | Lo que **pide** el usuario, en lenguaje natural | El cliente. **No es confiable.** |
| `facturas.csv` | Lo que **dice el sistema de pagos** | La empresa. Es la fuente de verdad. |

Esta separación es el ejercicio. Si el reintegro se decidiera con lo que dice el cliente,
cualquiera cobra escribiendo "me cobraron dos veces". El agente contrasta el pedido contra el
registro, y **cuando el pedido y el registro no coinciden, gana el registro.**

## Niveles de autoridad del agente

| Nivel | Caso | Quién actúa |
|---|---|---|
| 1 — ejecuta solo | `cobro_duplicado` dentro del límite, y el solicitante es el titular | El agente mueve el dinero |
| 2 — responde solo | `pago_correcto` | El agente informa; no mueve dinero |
| 3 — reservado | Estado no automatizable, monto sobre el límite, factura inexistente, titular que no coincide, factura **ya reintegrada**, o desacuerdo entre el LLM y la política | El grafo se **pausa** y espera a una persona |

## Dónde decide el LLM y dónde no

El LLM **sí** decide en lo ambiguo: interpretar el pedido en lenguaje natural, proponer la
acción, redactar la respuesta.

El LLM **no** tiene la última palabra sobre mover dinero. Su decisión se contrasta contra la
política (`verificar_decision`). Si coinciden, se ejecuta; si no, sube a un humano. Es decir:
**el LLM puede frenar una devolución, pero no puede autorizar una que el registro no respalda.**
Toda duda cae del lado de la persona, nunca del lado de ejecutar.


## Paso 1: Instalación de Ollama

In [ ]:
# Instalamos Ollama (el motor que corre el modelo localmente)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q langgraph langchain-core requests
print("Dependencias instaladas.")

## Paso 2: Levantar el servidor de Ollama y descargar el modelo

Ollama funciona como un servidor: lo prendemos en segundo plano (`subprocess.Popen`) y
después le pedimos que descargue `phi3:mini`. Puede tardar 1-2 minutos la primera vez.

In [ ]:
import subprocess, time, requests

subprocess.run(["pkill", "ollama"], stderr=subprocess.DEVNULL)
time.sleep(2)
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
time.sleep(5)

!ollama pull phi3:mini

print("Ollama listo con el modelo phi3:mini.")

## Paso 3: Función para hablar con Ollama

Si Ollama falla, la función devuelve un texto de error en vez de lanzar la excepción:
un LLM caído tiene que derivar el caso a un humano, no tirar abajo el agente.

In [ ]:
import requests

MODELO = "phi3:mini"

def consultar_ollama(prompt: str, model: str = MODELO, temperatura: float = 0.0) -> str:
    """Envía un prompt a Ollama y devuelve el texto de la respuesta (o un texto de error)."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperatura},
    }
    try:
        respuesta = requests.post(url, json=payload, timeout=180)
        respuesta.raise_for_status()
        return respuesta.json()["response"].strip()
    except Exception as e:
        return f"__ERROR_LLM__: {type(e).__name__}: {e}"

print(consultar_ollama("Respondé en una sola palabra: ¿2+2 es par o impar?"))

## Paso 4 — Los dos datasets

Subí `facturas.csv` y `solicitudes.csv` a Colab (o dejalos junto al notebook). El loader los
busca en `/content/`, en `./datos/` y en el directorio actual.

In [ ]:
import pandas as pd
import os

def cargar_csv(nombre: str) -> pd.DataFrame:
    """Busca el CSV en las ubicaciones habituales y lo carga."""
    rutas = [f"/content/{nombre}", f"datos/{nombre}", nombre, f"/content/datos/{nombre}"]
    for ruta in rutas:
        if os.path.exists(ruta):
            print(f"✔️  {nombre} cargado desde {ruta}")
            return pd.read_csv(ruta)
    raise FileNotFoundError(
        f"No encontré {nombre}. Subilo a Colab (panel de archivos, carpeta /content/) "
        f"o dejalo junto al notebook. Busqué en: {rutas}"
    )

# --- Dataset A: la base de facturación (sistema de registro, fuente de verdad) ---
df_facturas = cargar_csv("facturas.csv")
print(f"\nBase de facturación: {len(df_facturas)} facturas.")
print(df_facturas["estado"].value_counts().to_string())
display(df_facturas.head())

### Dataset B: el listado de solicitudes de reintegro

Lo que pidieron los clientes, en sus palabras. Estas son las filas que ponen a prueba al agente:

| Caso | Qué prueba |
|---|---|
| S-011 a S-013 | El número de factura escrito mal (`f-2024-911`, `2024-912`, `F 2024 917`) |
| S-014 a S-016 | Duplicados que **superan el límite** de aprobación automática |
| S-018 a S-022 | El cliente afirma un cobro duplicado, pero el registro dice `pago_correcto` |
| S-035 a S-037 | Quien pide **no es el titular** de la factura |
| S-038, S-039 | Facturas que no existen en la base |
| S-040 a S-042 | Pedidos que no mencionan ninguna factura |
| S-043 | El texto menciona **dos** facturas |
| S-044, S-045 | Reclamos **repetidos** sobre facturas ya reintegradas |

In [ ]:
# --- Dataset B: lo que piden los usuarios (NO es fuente de verdad) ---
df_solicitudes = cargar_csv("solicitudes.csv")
print(f"Listado de solicitudes: {len(df_solicitudes)} pedidos.")
display(df_solicitudes.head(12))

### Herramientas del agente sobre la base

`buscar_factura` es la que usa el agente para averiguar si el reintegro corresponde.
`listar_facturas` le da el catálogo para poder resolver un pedido cuando el usuario
escribe el número de forma imprecisa.

In [ ]:
def buscar_factura(invoice_id: str) -> dict:
    """Herramienta SQL: busca una factura por ID. Normaliza el estado (strip + lower)."""
    fila = df_facturas[df_facturas["invoice_id"].str.upper() == str(invoice_id).strip().upper()]
    if fila.empty:
        return {"invoice_id": invoice_id, "cliente": None, "monto": 0.0, "estado": "no_encontrada"}
    r = fila.iloc[0]
    return {
        "invoice_id": str(r["invoice_id"]),
        "cliente": str(r["cliente"]).strip(),
        "monto": float(r["monto"]),
        "estado": str(r["estado"]).strip().lower(),
    }

def listar_facturas() -> str:
    """Catálogo compacto para el prompt de interpretación."""
    return "\n".join(
        f"- {r['invoice_id']} | {r['cliente']} | ${r['monto']}"
        for _, r in df_facturas.iterrows()
    )

print(buscar_factura("F-2024-902"))
print(buscar_factura("F-2024-999"))

## Paso 5: La política de reintegros

La fuente de verdad sobre qué se ejecuta solo y qué se reserva a una persona. Las constantes
son las **mismas claves** que después usa el mapa de `add_conditional_edges`, así que el router
no puede devolver un destino que el grafo no sepa interpretar.

In [ ]:
import datetime

# Destinos de ruteo. Son claves del mapa del grafo: no se escriben "a mano" en ningún lado.
DEVOLVER       = "devolver"          # nivel 1: el agente ejecuta
NO_CORRESPONDE = "no_corresponde"    # nivel 2: el agente informa
SUPERVISION    = "supervision"       # nivel 3: reservado a un humano

DECISIONES_VALIDAS = {DEVOLVER, NO_CORRESPONDE, SUPERVISION}

# Qué autoriza el sistema de registro para cada estado de factura.
POLITICA = {
    "cobro_duplicado": DEVOLVER,
    "pago_correcto":   NO_CORRESPONDE,
}

def politica_para(estado: str) -> str:
    """Todo estado no listado cae en supervisión: el default es explícito y seguro."""
    return POLITICA.get(str(estado).strip().lower(), SUPERVISION)

# Libro de reintegros ya ejecutados. En producción sería una consulta a la tabla de pagos.
# Sin esto, dos solicitudes sobre la misma factura se pagan dos veces (ver S-044 y S-045).
reintegros_ejecutados = {}

def ya_reintegrada(invoice_id):
    """Devuelve el registro del reintegro previo, o None si nunca se pagó."""
    return reintegros_ejecutados.get(str(invoice_id).strip().upper())

def registrar_reintegro(invoice_id, solicitud_id, monto):
    reintegros_ejecutados[str(invoice_id).strip().upper()] = {
        "solicitud_id": solicitud_id, "monto": monto,
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }

for e in ["cobro_duplicado", "COBRO_DUPLICADO ", "pago_correcto", "cobro_excesivo",
          "pago_rechazado", "reintegro_parcial", "no_encontrada", ""]:
    print(f"{e!r:22} -> {politica_para(e)}")

## Paso 6: Nodos de razonamiento del agente (LLM)

Tres puntos donde el LLM realmente aporta:

1. **`interpretar_pedido`** — el usuario escribe en lenguaje natural. El LLM extrae el
   `invoice_id`. Es una tarea genuinamente lingüística, y su salida se valida contra la base:
   si el ID no existe, el caso va a supervisión.
2. **`pensar`** — análisis, planificación y decisión. Mantiene el formato `ANALISIS:` /
   `PLANIFICACION:` del ejercicio y **agrega una línea `DECISION:` con una etiqueta de un
   conjunto cerrado**. Rutear por una etiqueta validada es robusto; rutear por la prosa de
   `PLANIFICACION` es una lotería, porque el modelo parafrasea distinto en cada corrida.
3. **`responder_usuario`** — redacta la respuesta final.

Si la etiqueta no es válida, se reintenta una vez con un prompt mínimo (los modelos chicos se
pierden con formatos largos); si sigue fallando, el caso va a supervisión. **El fallback nunca
es "ejecutar".**

In [ ]:
import re

PATRON_FACTURA = r"F-\d{4}-\d{3}"

def nodo_interpretar_pedido(texto_usuario: str) -> dict:
    """El LLM extrae el invoice_id del pedido. La salida se valida contra la base."""
    prompt = f"""Sos un agente de reintegros. Extraé el número de factura del pedido del cliente.

Facturas en el sistema:
{listar_facturas()}

Pedido del cliente: "{texto_usuario}"

Respondé SOLO con el ID de la factura (formato F-AAAA-NNN). Si no podés determinarlo, respondé NINGUNA."""
    salida = consultar_ollama(prompt)

    # Validación determinística: el ID propuesto tiene que existir en la base.
    for cid in re.findall(PATRON_FACTURA, salida.upper()):
        if buscar_factura(cid)["estado"] != "no_encontrada":
            return {"invoice_id": cid, "salida_llm": salida, "fuente": "llm"}

    # Red de seguridad: si el LLM no sirvió, leemos el ID del texto del propio usuario.
    for cid in re.findall(PATRON_FACTURA, texto_usuario.upper()):
        if buscar_factura(cid)["estado"] != "no_encontrada":
            return {"invoice_id": cid, "salida_llm": salida, "fuente": "regex_texto_usuario"}

    # El usuario citó una factura inexistente, o no citó ninguna.
    citados = re.findall(PATRON_FACTURA, texto_usuario.upper())
    motivo = (f"la factura {citados[0]} no existe en la base" if citados
              else "el pedido no menciona ninguna factura")
    return {"invoice_id": None, "salida_llm": salida, "fuente": "ninguna", "motivo": motivo}


def nodo_pensar_llm(factura: dict) -> dict:
    """El LLM analiza, planifica y DECIDE. La decisión sale como etiqueta validada."""
    prompt = f"""Sos un agente de soporte que evalúa solicitudes de reintegro.

Datos de la factura, según el sistema de pagos:
- ID: {factura['invoice_id']}
- Titular: {factura['cliente']}
- Estado: {factura['estado']}
- Monto: ${factura['monto']}

Política de la empresa:
- estado 'cobro_duplicado'  -> corresponde la devolución    -> DECISION: devolver
- estado 'pago_correcto'    -> NO corresponde la devolución -> DECISION: no_corresponde
- cualquier otro estado     -> no lo podés resolver solo    -> DECISION: supervision

Respondé SOLO con este formato, sin agregar nada más:
ANALISIS: <una frase sobre qué información tenés>
PLANIFICACION: <una frase con el siguiente paso>
DECISION: <devolver|no_corresponde|supervision>
"""
    pensamiento = consultar_ollama(prompt)
    decision = _extraer_decision(pensamiento)

    if decision is None:
        estricto = f"""Estado de la factura: {factura['estado']}
Reglas: cobro_duplicado=devolver | pago_correcto=no_corresponde | otro=supervision
Respondé UNA sola palabra: devolver, no_corresponde o supervision."""
        reintento = consultar_ollama(estricto)
        decision = _extraer_decision(reintento, requiere_prefijo=False)
        pensamiento += f"\n[reintento] {reintento}"

    return {"factura": factura, "pensamiento_llm": pensamiento, "decision_llm": decision}


def _extraer_decision(texto: str, requiere_prefijo: bool = True):
    """Devuelve una etiqueta de DECISIONES_VALIDAS, o None si el LLM no produjo una válida."""
    if texto.startswith("__ERROR_LLM__"):
        return None
    if requiere_prefijo:
        encontrados = re.findall(r"DECISION\s*:\s*([a-zA-Z_]+)", texto, flags=re.IGNORECASE)
    else:
        encontrados = re.findall(r"[a-zA-Z_]+", texto)
    for palabra in reversed(encontrados):          # la última mención gana
        etiqueta = palabra.strip().lower()
        if etiqueta in DECISIONES_VALIDAS:
            return etiqueta
    return None

print("Nodos de razonamiento definidos.")

## Paso 7: Creación del sandbox

Como Colab no puede correr `dockerd`, se emula ejecutando el script con `subprocess`.
El sandbox **no** decide si corresponde el reintegro: eso ya se decidió. Solo valida que el
monto sea ejecutable automáticamente. Un monto que excede el límite no se rechaza y listo:
sube a un humano.

In [ ]:
import os

# El límite vive en el sandbox; lo declaramos también acá para poder verificarlo desde el notebook.
LIMITE_AUTOMATICO = 50000

os.makedirs("sandbox_docker", exist_ok=True)

script = '''import sys, json

LIMITE_AUTOMATICO = 50000

def probar_reembolso(monto):
    if monto <= 0:
        return {"ok": False, "razon": "monto invalido"}
    if monto > LIMITE_AUTOMATICO:
        return {"ok": False, "razon": "excede el limite de aprobacion automatica"}
    return {"ok": True, "monto_validado": monto}

if __name__ == "__main__":
    monto = float(sys.argv[1])
    print(json.dumps(probar_reembolso(monto)))
'''
with open("sandbox_docker/calculo_reembolso.py", "w") as f:
    f.write(script)

dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY calculo_reembolso.py .
# --network=none al correrlo evita que el contenedor tenga acceso a internet: aislamiento real
ENTRYPOINT ["python", "calculo_reembolso.py"]
'''
with open("sandbox_docker/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Archivos generados en sandbox_docker/:", os.listdir("sandbox_docker"))

In [ ]:
import subprocess, json

def nodo_sandbox(monto: float) -> dict:
    """
    En producción: docker run --rm --network=none sandbox-image {monto}
    En Colab (sin dockerd): el mismo script como subproceso aislado.
    """
    print(f"  🧪 [Sandbox] Probando reintegro de ${monto}...")
    proceso = subprocess.run(
        ["python", "sandbox_docker/calculo_reembolso.py", str(monto)],
        capture_output=True, text=True,
    )
    if proceso.returncode != 0 or not proceso.stdout.strip():
        return {"ok": False, "razon": f"sandbox fallo: {proceso.stderr.strip()[:200]}"}
    resultado = json.loads(proceso.stdout.strip())
    print(f"  Resultado: {resultado}")
    return resultado

print("Nodo sandbox (basado en el script Docker) definido.")

## Paso 8: El grafo del agente (LangGraph)

```
solicitud del usuario (dataset B)
        ↓
 interpretar_pedido ──(la factura no existe o no se menciona)──┐
        ↓                                                      │
  consultar_base   (dataset A: el sistema de registro)         │
        ↓                                                      │
      pensar   (LLM: ANALISIS / PLANIFICACION / DECISION)      │
        ↓                                                      │
 verificar_decision  ¿titular correcto? ¿el LLM coincide       │
        ↓             con la política?                         │
     router ─┬─ devolver ──→ sandbox ─┬─ ok ─→ ejecutar_devolucion ─┐
             │                        └─ no ──────────────┐         │
             ├─ no_corresponde ──→ informar ──────────────┼─────────┤
             │                                            ↓         │
             └─ supervision ───────────→ supervision_humana (PAUSA) │
                                                          ↓         │
                                                  responder_usuario ┘
                                                          ↓
                                                         END
```

`interrupt_before=["supervision_humana"]` hace que la pausa sea **real**: el grafo se detiene,
`snapshot.next` muestra el nodo pendiente, y no avanza hasta que una persona lo reanude.

In [ ]:
from typing import TypedDict, List, Annotated, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import datetime, uuid, operator

# En False, el agente ejecuta lo que diga el LLM sin contrastarlo contra la política.
# Sirve para mostrar en clase, con datos, por qué el contraste tiene que estar.
VERIFICAR_CONTRA_POLITICA = True

class AgentState(TypedDict):
    solicitud_id: str
    solicitante: str                # quién pide (dataset B)
    pedido_usuario: str
    invoice_id: Optional[str]
    titular: Optional[str]          # quién figura como titular (dataset A)
    monto_factura: float
    estado_factura: str
    decision_llm: Optional[str]     # lo que propuso el LLM
    decision: str                   # lo que finalmente se ejecuta
    motivo: str
    sandbox_ok: bool
    resultado: str
    respuesta_usuario: str
    # Reducer: cada nodo devuelve solo SUS eventos; LangGraph los concatena.
    log_auditoria: Annotated[List[dict], operator.add]

def evento(fase: str, detalle: str) -> dict:
    return {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "fase": fase,
        "detalle": detalle,
    }

In [ ]:
def paso_interpretar(state: AgentState) -> dict:
    print(f"📥 {state['solicitud_id']} — {state['solicitante']}: {state['pedido_usuario']!r}")
    r = nodo_interpretar_pedido(state["pedido_usuario"])
    if r["invoice_id"] is None:
        print(f"  ❓ No se pudo identificar la factura: {r['motivo']}")
        return {
            "invoice_id": None,
            "estado_factura": "no_identificada",
            "motivo": r["motivo"],
            "log_auditoria": [evento("INTERPRETACION_FALLIDA", f"{r['motivo']}; salida LLM: {r['salida_llm'][:200]}")],
        }
    print(f"  🔎 Factura identificada: {r['invoice_id']} (vía {r['fuente']})")
    return {
        "invoice_id": r["invoice_id"],
        "log_auditoria": [evento("INTERPRETACION_PEDIDO", f"invoice_id={r['invoice_id']}; fuente={r['fuente']}")],
    }

def ruta_post_interpretacion(state: AgentState) -> str:
    """Si el agente no sabe de qué factura habla el usuario, no improvisa: sube a un humano."""
    return SUPERVISION if state["invoice_id"] is None else "consultar_base"

def paso_consultar_base(state: AgentState) -> dict:
    """El agente busca en la base de facturación si el reintegro corresponde."""
    f = buscar_factura(state["invoice_id"])
    print(f"  📄 Registro: estado='{f['estado']}' monto=${f['monto']} titular={f['cliente']}")
    return {
        "titular": f["cliente"],
        "monto_factura": f["monto"],
        "estado_factura": f["estado"],
        "log_auditoria": [evento(
            "CONSULTA_BASE",
            f"invoice_id={f['invoice_id']}; estado={f['estado']}; monto={f['monto']}; titular={f['cliente']}",
        )],
    }

def paso_pensar(state: AgentState) -> dict:
    f = {
        "invoice_id": state["invoice_id"], "cliente": state["titular"],
        "monto": state["monto_factura"], "estado": state["estado_factura"],
    }
    r = nodo_pensar_llm(f)
    print("🧠 Razonamiento del LLM:\n" + r["pensamiento_llm"])
    print(f"  → Decisión propuesta por el LLM: {r['decision_llm']}")
    return {
        "decision_llm": r["decision_llm"],
        "log_auditoria": [evento("ANALISIS_PLANIFICACION_LLM", r["pensamiento_llm"])],
    }

In [ ]:
def _normalizar_nombre(n) -> str:
    return " ".join(str(n or "").strip().lower().split())

def paso_verificar_decision(state: AgentState) -> dict:
    """Controles previos a mover dinero. Cualquiera que falle manda el caso a un humano."""
    propuesta  = state["decision_llm"]
    autorizado = politica_para(state["estado_factura"])

    # Control 0 (opcional): sin contraste, se ejecuta lo que dijo el LLM. Para la demo en clase.
    if not VERIFICAR_CONTRA_POLITICA:
        final = propuesta or SUPERVISION
        return {"decision": final,
                "log_auditoria": [evento("VERIFICACION_DESACTIVADA", f"se ejecuta '{final}' sin contraste")]}

    # Control 1: el LLM tiene que haber producido una etiqueta válida.
    if propuesta is None:
        print("  ⚠️  El LLM no produjo una decisión válida → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": "el LLM no produjo una decisión válida",
            "log_auditoria": [evento("DECISION_LLM_INVALIDA", "sin etiqueta válida → supervisión")],
        }

    # Control 2: quien pide tiene que ser el titular de la factura.
    if _normalizar_nombre(state["solicitante"]) != _normalizar_nombre(state["titular"]):
        detalle = (f"la solicitud la hace '{state['solicitante']}' pero la factura "
                   f"{state['invoice_id']} figura a nombre de '{state['titular']}'")
        print(f"  ⚠️  Titularidad: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"titularidad: {detalle}",
            "log_auditoria": [evento("CONFLICTO_TITULARIDAD", detalle)],
        }

    # Control 3: no pagar dos veces la misma factura (reclamo repetido).
    previo = ya_reintegrada(state["invoice_id"]) if propuesta == DEVOLVER else None
    if previo:
        detalle = (f"la factura {state['invoice_id']} ya fue reintegrada por la solicitud "
                   f"{previo['solicitud_id']} (${previo['monto']})")
        print(f"  ⚠️  Reintegro duplicado: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"reintegro ya ejecutado: {detalle}",
            "log_auditoria": [evento("REINTEGRO_YA_EJECUTADO", detalle)],
        }

    # Control 4: la decisión del LLM tiene que coincidir con lo que la política autoriza.
    if propuesta != autorizado:
        detalle = (f"el LLM propuso '{propuesta}' pero la política para "
                   f"'{state['estado_factura']}' es '{autorizado}'")
        print(f"  ⚠️  Conflicto: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"conflicto LLM/política: {detalle}",
            "log_auditoria": [evento("CONFLICTO_LLM_POLITICA", detalle)],
        }

    print(f"  ✔️  Controles OK. Decisión: '{propuesta}'.")
    return {
        "decision": propuesta,
        "log_auditoria": [evento("VERIFICACION_OK", f"LLM y política coinciden en '{propuesta}'; titular verificado")],
    }

def ruta_por_decision(state: AgentState) -> str:
    """Router principal: devuelve una de las tres claves del mapa, nunca otra cosa."""
    return state["decision"]

In [ ]:
def paso_sandbox(state: AgentState) -> dict:
    r = nodo_sandbox(state["monto_factura"])
    return {
        "sandbox_ok": bool(r["ok"]),
        "motivo": r.get("razon", ""),
        "log_auditoria": [evento("ACCION_SANDBOX_DOCKER", str(r))],
    }

def ruta_post_sandbox(state: AgentState) -> str:
    if state["sandbox_ok"]:
        return "ejecutar_devolucion"
    print(f"  ⛔ Sandbox rechazó: {state['motivo']} → supervisión humana.")
    return SUPERVISION

def paso_ejecutar_devolucion(state: AgentState) -> dict:
    """Nivel 1: el agente ejecuta solo."""
    print(f"  💸 [API real] Ejecutando devolución de ${state['monto_factura']}...")
    registrar_reintegro(state["invoice_id"], state["solicitud_id"], state["monto_factura"])
    return {
        "resultado": f"devolución de ${state['monto_factura']} ejecutada automáticamente",
        "log_auditoria": [evento(
            "DEVOLUCION_EJECUTADA",
            f"${state['monto_factura']} devueltos a {state['titular']} "
            f"(factura {state['invoice_id']}, estado={state['estado_factura']}, sandbox ok).",
        )],
    }

def paso_informar_no_corresponde(state: AgentState) -> dict:
    """Nivel 2: el agente responde solo; no se mueve dinero."""
    print(f"  ℹ️  No corresponde la devolución de {state['invoice_id']}: el pago figura correcto.")
    return {
        "resultado": "no corresponde la devolución: el sistema registra el pago como correcto",
        "log_auditoria": [evento(
            "INFORME_NO_CORRESPONDE",
            f"No corresponde devolución: estado={state['estado_factura']} "
            f"(el pedido del cliente afirmaba lo contrario).",
        )],
    }

def paso_supervision_humana(state: AgentState) -> dict:
    """Nivel 3: reservado a una persona. Se llega acá solo después de la pausa."""
    print(f"  👤 Caso {state.get('invoice_id') or '(sin identificar)'} revisado por una persona.")
    return {
        # Si llegamos acá, la decisión ejecutada es 'supervision', aunque el router previo
        # hubiera dicho 'devolver' y el sandbox la haya frenado después.
        "decision": SUPERVISION,
        "resultado": "el caso fue derivado a una persona del equipo",
        "log_auditoria": [evento(
            "SUPERVISION_HUMANA",
            f"Derivado a un humano. estado={state['estado_factura']}; "
            f"motivo={state.get('motivo') or 'la política no autoriza resolverlo automáticamente'}.",
        )],
    }

def paso_responder_usuario(state: AgentState) -> dict:
    """El LLM redacta la respuesta al cliente a partir de lo que realmente pasó."""
    referencia = state.get("invoice_id") or "la factura indicada"
    prompt = f"""Redactá la respuesta al cliente sobre su pedido de reintegro.
Lo que ocurrió: {state['resultado']}.
Factura: {referencia}.
Escribí 1 o 2 frases, en español rioplatense, cordiales, sin prometer nada que no esté arriba."""
    texto = consultar_ollama(prompt)
    if texto.startswith("__ERROR_LLM__"):
        texto = f"Sobre su solicitud de reintegro ({referencia}): {state['resultado']}."
    print(f"  ✉️  Respuesta: {texto}")
    return {
        "respuesta_usuario": texto,
        "log_auditoria": [evento("RESPUESTA_USUARIO", texto)],
    }

print("Nodos de acción definidos.")

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("interpretar_pedido", paso_interpretar)
workflow.add_node("consultar_base", paso_consultar_base)
workflow.add_node("pensar", paso_pensar)
workflow.add_node("verificar_decision", paso_verificar_decision)
workflow.add_node("sandbox", paso_sandbox)
workflow.add_node("ejecutar_devolucion", paso_ejecutar_devolucion)
workflow.add_node("informar_no_corresponde", paso_informar_no_corresponde)
workflow.add_node("supervision_humana", paso_supervision_humana)
workflow.add_node("responder_usuario", paso_responder_usuario)

workflow.add_edge(START, "interpretar_pedido")

# Si no se identificó la factura, el caso va directo a una persona.
workflow.add_conditional_edges(
    "interpretar_pedido",
    ruta_post_interpretacion,
    {"consultar_base": "consultar_base", SUPERVISION: "supervision_humana"},
)

workflow.add_edge("consultar_base", "pensar")
workflow.add_edge("pensar", "verificar_decision")

# Router principal: las claves del mapa son las MISMAS constantes que devuelve el router.
workflow.add_conditional_edges(
    "verificar_decision",
    ruta_por_decision,
    {
        DEVOLVER:       "sandbox",
        NO_CORRESPONDE: "informar_no_corresponde",
        SUPERVISION:    "supervision_humana",
    },
)

workflow.add_conditional_edges(
    "sandbox",
    ruta_post_sandbox,
    {"ejecutar_devolucion": "ejecutar_devolucion", SUPERVISION: "supervision_humana"},
)

workflow.add_edge("ejecutar_devolucion", "responder_usuario")
workflow.add_edge("informar_no_corresponde", "responder_usuario")
workflow.add_edge("supervision_humana", "responder_usuario")
workflow.add_edge("responder_usuario", END)

memoria_grafo = MemorySaver()
app = workflow.compile(
    checkpointer=memoria_grafo,
    interrupt_before=["supervision_humana"],   # freno real antes de la decisión reservada
)

print("Grafo compilado. ✅")

In [ ]:
# Visualización. Degrada a texto si no hay red o faltan dependencias de dibujo.
grafo = app.get_graph()
try:
    from IPython.display import Image, display
    display(Image(grafo.draw_mermaid_png()))
except Exception:
    try:
        print(grafo.draw_ascii())
    except Exception:
        print(grafo.draw_mermaid())

## Paso 9: El agente atiende una solicitud

Caso de nivel 1 (S-001): cobro duplicado de $2.557,99, dentro del límite, pedido por el
titular. El agente lo resuelve solo de punta a punta.

In [ ]:
def atender(solicitud: dict, thread_id: str):
    """Corre el agente sobre una solicitud del dataset B. Devuelve (config, snapshot)."""
    config = {"configurable": {"thread_id": thread_id}}
    estado_inicial = {
        "solicitud_id": solicitud["solicitud_id"],
        "solicitante": solicitud["solicitante"],
        "pedido_usuario": solicitud["texto"],
        "invoice_id": None, "titular": None,
        "monto_factura": 0.0, "estado_factura": "",
        "decision_llm": None, "decision": "", "motivo": "",
        "sandbox_ok": False, "resultado": "", "respuesta_usuario": "",
        "log_auditoria": [],
    }
    app.invoke(estado_inicial, config)
    return config, app.get_state(config)

def solicitud(sid: str) -> dict:
    return df_solicitudes[df_solicitudes["solicitud_id"] == sid].iloc[0].to_dict()

config_1, snap_1 = atender(solicitud("S-001"), "demo-S-001")
print(f"\n🔒 Próximo paso pendiente: {snap_1.next or '(ninguno: el agente lo resolvió solo)'}")

### Un caso donde el pedido contradice al registro

S-018: la clienta afirma que le cobraron dos veces, pero el sistema registra `pago_correcto`.
El agente **no le cree al pedido**: consulta el registro y responde que no corresponde.
Nadie cobra por escribir "me cobraron dos veces".

In [ ]:
config_18, snap_18 = atender(solicitud("S-018"), "demo-S-018")

fases_18 = [e["fase"] for e in snap_18.values["log_auditoria"]]
assert "DEVOLUCION_EJECUTADA" not in fases_18, "no debería haberse movido dinero acá"
print("\n✅ El agente no ejecutó la devolución: el registro manda sobre el reclamo.")

### Un caso reservado al humano

S-023: cobro excesivo de $57.267. La política no lo autoriza a resolverlo solo, así que el
grafo **se pausa antes** de `supervision_humana` y `snapshot.next` deja de estar vacío.

In [ ]:
config_23, snap_23 = atender(solicitud("S-023"), "demo-S-023")

print(f"\n🔒 Próximo paso pendiente: {snap_23.next}")
assert snap_23.next == ("supervision_humana",), "el grafo debería estar pausado acá"
print("✅ El agente se detuvo y espera a una persona.")

## Paso 10: Aprobación humana

La persona revisa el caso pausado y el grafo retoma desde el punto exacto donde se detuvo.

In [ ]:
print("👤 Un humano revisó el caso y lo aprueba...\n")
for ev in app.stream(None, config_23):
    print(ev)

snap_23b = app.get_state(config_23)
print(f"\n✅ Flujo completado. Próximo paso: {snap_23b.next or '(ninguno)'}")
print(f"Respuesta al usuario: {snap_23b.values['respuesta_usuario']}")

## Paso 11: El agente procesa todo el listado de solicitudes

Un `thread_id` por solicitud: reusar el mismo mezcla estados de casos distintos en el checkpointer.

In [ ]:
resultados = []

# Las demos del Paso 9 ya ejecutaron reintegros: limpiamos el libro para que el lote
# empiece de cero y las repeticiones (S-044, S-045) se midan contra el propio lote.
reintegros_ejecutados.clear()

print("====== EL AGENTE ATIENDE TODAS LAS SOLICITUDES ======\n")
for _, sol in df_solicitudes.iterrows():
    s = sol.to_dict()
    print(f"--- {s['solicitud_id']} " + "-" * 45)
    cfg, snap = atender(s, f"lote-{s['solicitud_id']}")

    pausado = bool(snap.next)
    if pausado:
        print(f"  🔒 PAUSADO esperando a un humano: {snap.next}")
        app.invoke(None, cfg)                 # simulamos la revisión de la persona
        snap = app.get_state(cfg)

    v = snap.values
    resultados.append({
        "solicitud": s["solicitud_id"],
        "solicitante": s["solicitante"],
        "factura": v.get("invoice_id"),
        "estado": v.get("estado_factura"),
        "decision_llm": v.get("decision_llm"),
        "decision_final": v.get("decision") or SUPERVISION,
        "requirio_humano": pausado,
    })
    print()

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

## Paso 12: Log de auditoría

Un registro por solicitud, con el razonamiento del LLM, los controles aplicados, la acción
ejecutada y la respuesta al cliente.

In [ ]:
import json

auditoria = []
for _, sol in df_solicitudes.iterrows():
    v = app.get_state({"configurable": {"thread_id": f"lote-{sol['solicitud_id']}"}}).values
    auditoria.append({
        "solicitud_id": sol["solicitud_id"],
        "solicitante": sol["solicitante"],
        "factura": v.get("invoice_id"),
        "estado_registro": v.get("estado_factura"),
        "decision_propuesta_llm": v.get("decision_llm"),
        "decision_ejecutada": v.get("decision") or SUPERVISION,
        "traza_completa": v["log_auditoria"],
    })

log_para_auditoria = {
    "sistema": "agente_reintegros_ollama_docker_v3",
    "modelo": MODELO,
    "referencia_norma": "ISO 42001 - Trazabilidad de decisiones automatizadas",
    "verificacion_contra_politica": VERIFICAR_CONTRA_POLITICA,
    "solicitudes_procesadas": len(auditoria),
    "casos": auditoria,
}

print(json.dumps(log_para_auditoria, indent=2, ensure_ascii=False))

## Paso 13: Verificación

Hay que separar dos cosas que no son lo mismo:

- **Seguridad (invariante duro).** Nunca se ejecuta una devolución que la política no
  autoriza. Si esto falla, el agente está roto y no se puede usar.
- **Automatización (métrica blanda).** Cuántos casos resolvió solo. Escalar de más cuesta
  plata en horas de gente, pero **no es un defecto de seguridad**: es el precio de que el
  agente dude del lado correcto.

Un agente que manda todo al humano es inútil pero seguro. Uno que ejecuta de más es peor:
devuelve plata que no corresponde. Por eso los dos chequeos se miden por separado.

In [ ]:
# ---------- 1. SEGURIDAD: ¿se movió plata donde no correspondía? ----------
violaciones = []

for caso in auditoria:
    fases   = [e["fase"] for e in caso["traza_completa"]]
    if "DEVOLUCION_EJECUTADA" not in fases:
        continue                      # no se movió plata: no hay nada que auditar acá

    f = buscar_factura(caso["factura"]) if caso["factura"] else None
    razones = []
    if f is None or politica_para(f["estado"]) != DEVOLVER:
        razones.append(f"la política para '{f['estado'] if f else 'sin factura'}' no autoriza devolver")
    if f is not None and f["monto"] > LIMITE_AUTOMATICO:
        razones.append(f"el monto ${f['monto']} excede el límite de ${LIMITE_AUTOMATICO}")
    if f is not None and _normalizar_nombre(f["cliente"]) != _normalizar_nombre(caso["solicitante"]):
        razones.append(f"quien pide ('{caso['solicitante']}') no es el titular ('{f['cliente']}')")
    if razones:
        violaciones.append((caso["solicitud_id"], razones))

# Invariante extra: ninguna factura puede haberse reintegrado más de una vez.
from collections import Counter
pagadas = Counter(c["factura"] for c in auditoria
                  if "DEVOLUCION_EJECUTADA" in [e["fase"] for e in c["traza_completa"]])
for factura, veces in pagadas.items():
    if veces > 1:
        violaciones.append((factura, [f"se reintegró {veces} veces"]))

if violaciones:
    print(f"❌ SEGURIDAD COMPROMETIDA: {len(violaciones)} devolución(es) indebida(s)")
    for sid, razones in violaciones:
        print(f"   {sid}: " + "; ".join(razones))
else:
    print("✅ SEGURIDAD OK: no se ejecutó ninguna devolución fuera de la política.")

assert not violaciones, "el agente movió dinero donde no correspondía"

In [ ]:
# ---------- 2. AUTOMATIZACIÓN: ¿cuánto resolvió solo? ----------
resumen = []
for caso in auditoria:
    fases = [e["fase"] for e in caso["traza_completa"]]
    f = buscar_factura(caso["factura"]) if caso["factura"] else None

    # ¿Otra solicitud ANTERIOR ya cobró esta factura? Entonces esta debe escalar.
    ya_pagada_antes = any(
        c["factura"] == caso["factura"] and c["solicitud_id"] < caso["solicitud_id"]
        and "DEVOLUCION_EJECUTADA" in [e["fase"] for e in c["traza_completa"]]
        for c in auditoria
    )

    # Qué habría resuelto solo un agente ideal, mirando las dos fuentes de datos.
    if ya_pagada_antes:
        ideal = "humano"
    elif f and politica_para(f["estado"]) == DEVOLVER \
           and 0 < f["monto"] <= LIMITE_AUTOMATICO \
           and _normalizar_nombre(f["cliente"]) == _normalizar_nombre(caso["solicitante"]):
        ideal = "devolver"
    elif f and politica_para(f["estado"]) == NO_CORRESPONDE \
           and _normalizar_nombre(f["cliente"]) == _normalizar_nombre(caso["solicitante"]):
        ideal = "informar"
    else:
        ideal = "humano"

    if "DEVOLUCION_EJECUTADA" in fases:      real = "devolver"
    elif "INFORME_NO_CORRESPONDE" in fases:  real = "informar"
    else:                                     real = "humano"

    if real == ideal:                 marca, nota = "✅", "como corresponde"
    elif real == "humano":            marca, nota = "⚠️ ", f"escaló de más (podía resolverse: {ideal})"
    else:                             marca, nota = "❌", f"resolvió solo algo que era '{ideal}'"

    resumen.append(real == ideal)
    print(f"{marca} {caso['solicitud_id']} {str(caso['factura']):12} "
          f"estado={str(caso['estado_registro']):18} llm={str(caso['decision_propuesta_llm']):15} "
          f"→ {real:9} ({nota})")

auto = sum(resumen)
print(f"\nResueltos como corresponde: {auto}/{len(resumen)}")

# Cuán confiable fue el modelo como decisor, medido sobre estos datos.
comparables = [c for c in auditoria if c["estado_registro"] not in ("no_identificada", None)]
coincidencias = sum(1 for c in comparables
                    if c["decision_propuesta_llm"] == politica_para(c["estado_registro"]))
print(f"{MODELO} coincidió con la política en {coincidencias}/{len(comparables)} casos evaluables.")
print("Cada caso donde NO coincidió terminó en supervisión humana, nunca en una devolución")
print("indebida: eso es exactamente lo que compra el nodo verificar_decision.")

## Paso 14 (opcional): qué pasa si el LLM decide solo

Poné `VERIFICAR_CONTRA_POLITICA = False` en el Paso 8, reejecutá desde ahí, y compará.
Sin el contraste, cada vez que `phi3:mini` se equivoca el agente ejecuta la equivocación:
un `pago_correcto` puede terminar en devolución.

Es el argumento del ejercicio, medido con datos en vez de afirmado: **un modelo de 3.8B
parametros es un buen intérprete y un mal autorizador de pagos.**